This notebook was run last on the following commit

In [ ]:
!git log -1

In [ ]:
%matplotlib inline

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotnine as gg
import numpy as np
import sys
import os

sys.path.append("/workspace")


from src.evaluation.kernel_evaluation import (
    compute_distance_matrix,
    process_and_align,
    wide_to_long,
)
import glob
import json
from essential.utils import PLOTNINE_DEFAULT_THEME

In [ ]:
SAVE_DPI = 600

In [ ]:
FONT_FAMILY = "Arial"

PLOTNINE_DEFAULT_THEME_2 = gg.theme(
    # Half-open frame: thin left + bottom axes only
    axis_line=gg.element_line(color="#333333", size=0.4),
    axis_line_x=gg.element_line(color="#333333", size=0.4),
    axis_line_y=gg.element_line(color="#333333", size=0.4),
    # Small outward ticks
    axis_ticks=gg.element_line(color="#333333", size=0.3),
    axis_ticks_length=3,
    axis_ticks_direction="out",
    axis_ticks_minor=gg.element_blank(),
    # Text
    axis_text=gg.element_text(size=6, family=FONT_FAMILY, color="#333333"),
    axis_title=gg.element_text(size=7, family=FONT_FAMILY, color="#333333", weight="normal"),
    title=gg.element_text(size=7, family=FONT_FAMILY, weight="bold"),
    # Clean background
    panel_background=gg.element_rect(fill="white", color="none"),
    panel_grid_major=gg.element_blank(),
    panel_grid_minor=gg.element_blank(),
    plot_background=gg.element_rect(fill="white", color="none"),
    panel_border=gg.element_blank(),
    # Sizing
    figure_size=(3, 2),
    # Legend
    legend_title=gg.element_text(size=6, family=FONT_FAMILY, color="#333333", weight="normal"),
    legend_text=gg.element_text(size=5, family=FONT_FAMILY, color="#333333"),
    legend_key_size=10,
    legend_key=gg.element_rect(fill="white", color="none"),
    legend_background=gg.element_rect(fill="white", color="#CCCCCC", size=0.3),
    legend_margin=0,
    legend_entry_spacing=2,
    # Facet strips
    strip_text=gg.element_text(size=6, family=FONT_FAMILY, color="#333333", weight="normal"),
    strip_background=gg.element_rect(fill="white", color="none"),
)

In [ ]:
SAVE_DIR = "/workspace/experiments/03262026_evaluation/figures"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
EXPERIMENT_NAME = "04032026"

In [ ]:
METHODS_TO_PLOT = [
    "fba_moma_default",
    "gene_graph_beta_1",
    "gene_graph_beta_auto",
    "fba_gene_graph_beta_1",
    "fba_gene_graph_beta_auto",
]

METHOD_RENAMER = {
    "fba_moma_default": "flux model",
    "gene_graph_beta_1": r"graph ($\beta=1$)",
    "gene_graph_beta_auto": r"graph ($\beta=\lambda_{\max}^{-1}$)",
    "fba_gene_graph_beta_1": r"flux-filt. graph ($\beta=1$)",
    "fba_gene_graph_beta_auto": r"flux-filt. graph ($\beta=\lambda_{\max}^{-1}$)",
}

METHOD_COLORS = {
    "flux model": "#4878CF",  # steel blue — standalone
    r"graph ($\beta=1$)": "#E07B54",  # terracotta
    r"graph ($\beta=\lambda_{\max}^{-1}$)": "#B5503A",  # darker terracotta
    r"flux-filt. graph ($\beta=1$)": "#6AAF6A",  # sage green
    r"flux-filt. graph ($\beta=\lambda_{\max}^{-1}$)": "#3D7A3D",  # darker sage
}

# Scatter plots

In [ ]:
tag_name = "fba_gene_graph_beta_1"
# tag_name = "fba_gene_graph_beta_auto"
dmat_path = f"/workspace/results/ecoli_rich_medium/pred/{tag_name}/distances.pkl"


pred_dist = pd.read_pickle(dmat_path)
target_dist = pd.read_pickle(f"/workspace/results/ecoli_rich_medium/targets/mmd_distances.pkl")

pred_dist_sub, target_dist_sub = process_and_align(pred_dist, target_dist)

# metric: percentile of distances in target for smallest elements in pred
pred_dist_sub_long = wide_to_long(
    pred_dist_sub,
    "distance_pred",
    "gene1",
    "gene2",
    remove_diagonal=True,
    remove_lower_triangle=True,
)
target_dist_sub_long = wide_to_long(
    target_dist_sub,
    "distance_target",
    "gene1",
    "gene2",
    remove_diagonal=True,
    remove_lower_triangle=True,
)
joint_long = pd.merge(pred_dist_sub_long, target_dist_sub_long, on=["gene1", "gene2"], how="inner")

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, metric="precomputed", init="random")
tsne_result = tsne.fit_transform(pred_dist)
plot_df = pd.DataFrame(tsne_result, index=pred_dist.columns, columns=["tsne_x", "tsne_y"])

fig = (
    gg.ggplot(plot_df, gg.aes(x="tsne_x", y="tsne_y"))
    + gg.geom_point(size=0.5)
    + gg.theme_classic()
    + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(
        x="TSNE1",
        y="TSNE2",
    )
    + gg.theme(
        figure_size=(2.5, 2),
    )
)
fig.save(os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_{tag_name}_tsne.png"), dpi=SAVE_DPI)
display(fig)


fig2 = (
    gg.ggplot(joint_long, gg.aes(x="distance_pred", y="distance_target"))
    + gg.geom_point(size=0.1)
    + gg.geom_hline(yintercept=0.1, color="red")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(
        x="metabolic distance",
        y="transcriptomic distance",
    )
    + gg.theme(
        figure_size=(2.5, 2),
    )
)
fig2.save(os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_{tag_name}_joint_distances.png"), dpi=SAVE_DPI)
display(fig2)

# General metrics of the metabolic distances

In [ ]:
from src.evaluation.kernel_evaluation import wide_to_long

nvals = np.linspace(10, 10000, 100)
# unqiue genes in top K pairs
plot_df = pd.DataFrame()
for m in METHODS_TO_PLOT:
    dmat_path = f"/workspace/results/ecoli_rich_medium/pred/{m}/distances.pkl"
    dmat = pd.read_pickle(dmat_path)

    for k in [10, 50, 100]:
        dmat_arr = dmat.to_numpy(copy=True)
        np.fill_diagonal(dmat_arr, np.inf)
        knn_idx = np.argsort(dmat_arr, axis=1)[:, :k]
        counts = np.bincount(knn_idx.flatten(), minlength=dmat.shape[1])
        gene_k_occurrences = (
            pd.Series(counts, index=dmat.columns).to_frame("occurence").assign(method=m, K=k)
        )
        plot_df = pd.concat([plot_df, gene_k_occurrences])
plot_df["method"] = plot_df["method"].map(METHOD_RENAMER)

In [ ]:
fig = (
    gg.ggplot(plot_df, gg.aes(x="method", y="occurence", fill="method"))
    + gg.geom_boxplot(
        outlier_size=0.8,
        outlier_shape="o",
        outlier_alpha=0.5,
        outlier_stroke=0.3,
    )
    + gg.facet_wrap("K", scales="free_y", labeller="label_both")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        legend_position="none",
        axis_text_x=gg.element_text(rotation=45, ha="right", size=5),
        figure_size=(4, 2),
    )
    + gg.labs(
        x="",
    )
    + gg.scale_fill_manual(values=METHOD_COLORS)
)
fig.save(os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_gene_k_occurrences.png"), dpi=SAVE_DPI)
fig

# Agreement with KEGG pathway annotations

In [ ]:
all_results = []
for m in METHODS_TO_PLOT:
    file_path = f"/workspace/results/ecoli_rich_medium/pred/{m}/kegg_metrics.csv"
    kegg_metrics = pd.read_csv(file_path).assign(method=m)
    all_results.append(kegg_metrics)
all_results = pd.concat(all_results)
all_results["method"] = all_results["method"].map(METHOD_RENAMER)

In [ ]:
fig = (
    gg.ggplot(all_results, gg.aes(x="factor(method)", y="distance_ratio", fill="method"))
    + gg.geom_boxplot(
        outlier_size=0.8,
        outlier_shape="o",
        outlier_alpha=0.5,
        outlier_stroke=0.3,
    )
    + gg.geom_hline(yintercept=1.0, color="black")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        legend_position="none",
        figure_size=(1.7, 2),
        axis_text_x=gg.element_text(rotation=45, ha="right", size=5),
    )
    + gg.labs(x="", y="KEGG distance ratio")
    + gg.scale_fill_manual(values=METHOD_COLORS)
)
fig.save(os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_kegg_distance_ratio.png"), dpi=SAVE_DPI)
fig

# Agreement with transcriptomic distances

In [ ]:
distance_metrics_files = glob.glob(
    "/workspace/results/ecoli_rich_medium/*/*/distance_metrics_mmd.json"
)
all_results = []
for f in distance_metrics_files:
    with open(f, "r") as f:
        results = json.load(f)
    all_results.append(results)

all_results_df = pd.DataFrame(all_results)

In [ ]:
K_values = [50, 100, 500, 1000]
SELECTED_COLUMNS = [f"target_dist_median_ratio_of_top_{K}_pred_pairs_to_global" for K in K_values]
renamer = {
    f"target_dist_median_ratio_of_top_{K}_pred_pairs_to_global": f"$K={K}$" for K in K_values
}


plot_df = (
    all_results_df.loc[all_results_df["tag"].isin(METHODS_TO_PLOT)]
    .set_index("tag")
    .loc[:, SELECTED_COLUMNS]
)
plot_df = (
    plot_df.loc[plot_df["target_dist_median_ratio_of_top_100_pred_pairs_to_global"].notna()]
    .stack()
    .to_frame("transcriptomic_distance_ratio")
    .reset_index()
    .rename(columns={"level_1": "_K"})
    .assign(
        K=lambda x: pd.Categorical(x["_K"].map(renamer), categories=renamer.values(), ordered=True),
        method=lambda x: x["tag"].map(METHOD_RENAMER),
    )
)
plot_df

In [ ]:
fig = (
    gg.ggplot(plot_df, gg.aes(x="K", y="transcriptomic_distance_ratio", fill="method"))
    + gg.geom_col(position="dodge")
    + gg.theme_minimal()
    + gg.geom_hline(yintercept=1.0, color="black")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        # legend_position="none",
        axis_text_x=gg.element_text(rotation=45, ha="right", size=5),
        figure_size=(4, 2),
    )
    + gg.scale_fill_manual(values=METHOD_COLORS)
    + gg.labs(x="", y="transcriptomic distance ratio")
)
fig.save(
    os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_transcriptomic_distance_ratio.png"), dpi=SAVE_DPI
)
fig

# LLM evaluation

In [ ]:
all_llm_relevance_evals = []
for model in METHODS_TO_PLOT:
    paths = glob.glob(f"/workspace/results/ecoli_rich_medium/pred/{model}/llm_relevance_run_*.json")
    llm_relevance_evals = []
    for path in paths:
        with open(path, "r") as f:
            data = json.load(f)
        llm_relevance_evals.append(data)

    llm_relevance_evals_df = pd.DataFrame(llm_relevance_evals).T.mean(axis=1)
    all_llm_relevance_evals.append(
        {
            "method": model,
            "relevance_mean": llm_relevance_evals_df.mean(),
            "relevance_std": llm_relevance_evals_df.std() / np.sqrt(len(llm_relevance_evals_df)),
        }
    )

all_llm_relevance_evals_df = pd.DataFrame(all_llm_relevance_evals)
all_llm_relevance_evals_df["method"] = all_llm_relevance_evals_df["method"].map(METHOD_RENAMER)

In [ ]:
fig = (
    gg.ggplot(
        all_llm_relevance_evals_df, gg.aes(x="factor(method)", y="relevance_mean", color="method")
    )
    + gg.geom_point()
    + gg.geom_errorbar(
        gg.aes(ymin="relevance_mean - relevance_std", ymax="relevance_mean + relevance_std"),
        width=0.2,
    )
    + gg.theme_minimal()
    + gg.coord_flip()
    + PLOTNINE_DEFAULT_THEME_2
    + gg.scale_color_manual(values=METHOD_COLORS)
    + gg.labs(x="", y="LLM relevance score")
    + gg.theme(
        figure_size=(3, 1.5),
        legend_position="none",
    )
)
fig.save(os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_llm_relevance_score.png"), dpi=SAVE_DPI)
fig

In [ ]:
all_mechanisms_df = []
for model in METHODS_TO_PLOT:
    paths = glob.glob(
        f"/workspace/results/ecoli_rich_medium/pred/{model}/llm_mechanism_mmd_run_*.json"
    )
    llm_mechanism_evals = []
    for path in paths:
        with open(path, "r") as f:
            data = json.load(f)
        mechanisms = pd.DataFrame(data).T.assign(
            run=path.split("_")[-1].split(".")[0], method=model
        )
        all_mechanisms_df.append(mechanisms.reset_index().rename(columns={"index": "gene_pair"}))
all_mechanisms_df = pd.concat(all_mechanisms_df)
all_mechanisms_df["is_relevant"] = (all_mechanisms_df["score"] >= 2.0).astype(float)
all_mechanisms_df["method"] = all_mechanisms_df["method"].map(METHOD_RENAMER)

In [ ]:
all_mechanisms_df_merged = (
    all_mechanisms_df.groupby(["gene_pair", "method"])["is_relevant"].mean().reset_index()
)
mechanism_scores = (
    all_mechanisms_df_merged.groupby("method")["is_relevant"].agg(["mean", "std"]).reset_index()
)
fig = (
    gg.ggplot(mechanism_scores, gg.aes(x="factor(method)", y="mean", color="method"))
    + gg.geom_point()
    + gg.theme_minimal()
    + gg.geom_errorbar(gg.aes(ymin="mean - std", ymax="mean + std"), width=0.2)
    + gg.coord_flip()
    + PLOTNINE_DEFAULT_THEME_2
    + gg.scale_color_manual(values=METHOD_COLORS)
    + gg.labs(x="", y="LLM mechanism score")
    + gg.theme(
        figure_size=(3, 1.5),
        legend_position="none",
    )
)
fig.save(os.path.join(SAVE_DIR, f"{EXPERIMENT_NAME}_llm_mechanism_score.png"), dpi=SAVE_DPI)
fig

In [ ]:
set1 = all_mechanisms_df_merged.loc[lambda x: x["method"] == "fba_gene_graph_beta_1"][
    "gene_pair"
].tolist()
set2 = all_mechanisms_df_merged.loc[lambda x: x["method"] == "fba_gene_graph_beta_auto"][
    "gene_pair"
].tolist()

len(set(set1) & set(set2))

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

adata.obs["umap_x"] = adata.obsm["X_umap"][:, 0]
adata.obs["umap_y"] = adata.obsm["X_umap"][:, 1]

In [ ]:
from essential.data import load_fitness_data

fitness_df = load_fitness_data().groupby("gene")[["T1", "T2", "T3", "T4"]].mean()
fitness_df

In [ ]:
# gene_list = ["fabI", "fabZ", "fabA", "fabG"]
gene_list = ["bioA", "bioF", "bioH"]  # Differential expression
for gene in gene_list:
    display(fitness_df.loc[gene])
    print()


adata_sub = adata[adata.obs["gene"].isin(gene_list)]
fig = (
    gg.ggplot(
        adata.obs,
        gg.aes(x="umap_x", y="umap_y"),
    )
    + gg.geom_point()
    + gg.geom_point(adata_sub.obs, gg.aes(x="umap_x", y="umap_y", color="gene"))
)
display(fig)

In [ ]:
adata_case

In [ ]:
# gene_list = ["fabI", "fabZ", "fabA", "fabG"]
# gene_list = ["bioA", "bioF", "bioH", "bioD", "bioC"]  # Differential expression, do not overthink fitness
gene_list = ["cdsA", "pssA"]
for gene in gene_list:
    display(fitness_df.loc[gene])
    print()


adata_sub = adata_case[adata_case.obs["gene"].isin(gene_list)]
fig = (
    gg.ggplot(
        adata_case.obs,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
    )
    + gg.geom_point()
    + gg.geom_point(
        adata_sub.obs, gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene")
    )
)
display(fig)

In [ ]:
fabI - fabZ

fabI looks like fabZ
